# VygotEA — Evaluation Notebook (v3)
**Paper:** *VygotEA: A Hybrid AI System for Adaptive Socioemotional Support in Students with ASD*  

## Estrutura do Notebook
| Célula | Conteúdo |
|--------|----------|
| 1 | Instalação de dependências |
| 2 | Carregamento da configuração (`vygotea_config.json`) |
| 3 | Geração de dados sintéticos (Pool A treino + Pool B teste) |
| 4 | Carregamento do dataset piloto real |
| 5 | Classificadores — treinamento e avaliação |
| 6 | Análise por classe (TF-IDF + LR) |
| 7 | Análise do mecanismo ZPD |
| 8 | Efetividade das estratégias de mediação |
| 9 | Correlação progresso × satisfação |
| 10 | Rubrica qualitativa de especialistas |
| 11 | Geração das figuras para publicação |
| 12 | Demo — interação individual |

> **Nota:** Todos os dados de configuração (perfis, templates, estratégias, dados piloto)
> estão em `vygotea_config.json`.


## Célula 1 — Instalação de Dependências

In [1]:
# dependências
!pip install scikit-learn scipy matplotlib seaborn pandas numpy -q


## Célula 2 — Imports e Configuração Global
Toda a configuração vive em `vygotea_config.json`; aqui apenas lemos e expondo as variáveis.


In [6]:
import json, random, warnings, textwrap
import numpy as np
import pandas as pd
from collections import defaultdict
from datetime import datetime, timedelta
from typing import Dict, List
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, accuracy_score, classification_report
from scipy.stats import pearsonr

import matplotlib
matplotlib.use('Agg')  # sem display — salva em arquivo
import matplotlib.pyplot as plt
import seaborn as sns
import os
import urllib

In [8]:
# Baixar arquivos externos
def download_data(file_name, file_url, save_dir="/content/sample_data"):
    file_path = os.path.join(save_dir, file_name)
    if not os.path.exists(file_path):
        try:
            urllib.request.urlretrieve(file_url, file_path)
            print(f"Download concluído: {file_path}")
        except Exception as e:
            print(f"Erro ao baixar o arquivo: {e}")
    return file_path

CONFIG_PATH = 'vygotea_config.json'

# Baixar dataset
file_url = "https://www.dropbox.com/scl/fi/e0rj47ko4gdjbee0bar9n/vygotea_config.json?rlkey=binog27zuzgjueh9kr1mbjv8u&dl=1"
caminho_do_arquivo_json = download_data(CONFIG_PATH, file_url)

In [10]:
# Carregar config
cfg = json.load(open("/content/sample_data/"+CONFIG_PATH))

PROFILES      = cfg['student_profiles']
TEMPLATES_TR  = cfg['templates_train']
TEMPLATES_TE  = cfg['templates_test']
MEDIATION     = cfg['mediation_strategies']
RESPONSES     = cfg['adaptive_responses']
SITUATIONS    = cfg['school_situations']
EMOTION_DIST  = cfg['emotion_distribution']
CONF_MU       = cfg['confidence_mu']
KW_DICT       = cfg['keyword_dict']
SIM           = cfg['simulation']
RUBRIC_CFG    = cfg['specialist_rubric']

random.seed(SIM['seed'])
np.random.seed(SIM['seed'])

print('Configuração carregada com sucesso.')
print(f"  Perfis de estudantes: {list(PROFILES.keys())}")
print(f"  Emoções: {list(EMOTION_DIST.keys())}")
print(f"  Seed: {SIM['seed']} | Train: {SIM['n_train']} | Test: {SIM['n_test']} | Pilot: {SIM['n_pilot']}")

Configuração carregada com sucesso.
  Perfis de estudantes: ['P1', 'P2', 'P3', 'P4', 'P5']
  Emoções: ['anxiety', 'frustration', 'calm', 'sadness', 'joy', 'fear']
  Seed: 42 | Train: 800 | Test: 200 | Pilot: 25


## Célula 3 — Geração de Dados Sintéticos
Pool A (treino, n=800) e Pool B (teste, n=200) com **templates lexicais distintos** para evitar data leakage.
Ver discussão metodológica no artigo, §3.8 e §6.2.


In [11]:
# ── Funções auxiliares de geração ────────────────────────────────────────────

def _engagement(profile: Dict) -> float:
    """Equação (3): Et = (H+A+C)/15 + ε,  ε ~ U(−0.1, 0.1)"""
    base = (profile['h'] + profile['a'] + profile['c']) / 15.0
    return float(np.clip(base + np.random.uniform(-0.1, 0.1), 0.0, 1.0))


def _adaptive_progress(profile: Dict, t: int, prev: float, alpha: float = None) -> float:
    """Equação (2): Pt = α·Et + (1−α)·Pt-1"""
    alpha = alpha or SIM['alpha_smoothing']
    return round(float(np.clip(alpha * _engagement(profile) + (1 - alpha) * prev, 0.0, 1.0)), 4)


def _zdp_classification(profile: Dict) -> str:
    """Classifica a interação como within_ZPD, NDR ou out_of_ZPD."""
    r = random.random()
    w = 0.55 + (profile['a'] - 1) * 0.03  # profis com maior regulação têm maior prob. within ZPD
    if r < w:          return 'within_ZPD'
    elif r < w + 0.128: return 'NDR'
    else:               return 'out_of_ZPD'


def generate_interactions(n: int, templates: Dict) -> pd.DataFrame:
    """
    Gera n interações sintéticas a partir dos templates fornecidos.
    Use TEMPLATES_TR para Pool A (treino) e TEMPLATES_TE para Pool B (teste).
    """
    # Constrói pool de emoções respeitando a distribuição alvo
    emotion_pool = []
    for em, frac in EMOTION_DIST.items():
        emotion_pool.extend([em] * round(frac * n))
    random.shuffle(emotion_pool)
    emotion_pool = (emotion_pool * 2)[:n]  # garante exatamente n amostras

    profile_keys = list(PROFILES.keys())
    prev_progress = {pk: (PROFILES[pk]['h'] + PROFILES[pk]['a'] + PROFILES[pk]['c']) / 15
                     for pk in profile_keys}
    interaction_count = defaultdict(int)
    rows = []

    for i, emotion in enumerate(emotion_pool):
        pk      = profile_keys[i % len(profile_keys)]
        profile = PROFILES[pk]
        situation = random.choice(SITUATIONS)

        # Mensagem gerada a partir do template
        message = random.choice(templates[emotion]).format(s=situation.replace('_', ' '))

        # Nível ZPD do perfil (estático por perfil)
        zdp_level = profile['n']

        # Estratégia de mediação (Tabela 2 do artigo)
        strategy = MEDIATION[emotion][zdp_level]

        # Progresso adaptativo (Eq. 2)
        t       = interaction_count[pk]
        progress = _adaptive_progress(profile, t, prev_progress[pk])
        prev_progress[pk] = progress
        interaction_count[pk] += 1

        # Satisfação simulada (μ=4.26, σ=0.38 — paper §4.1)
        satisfaction = float(np.clip(
            np.random.normal(SIM['satisfaction_mu'], SIM['satisfaction_sigma']), 1.0, 5.0))

        # Confiança do classificador (varia por emoção — Tabela 6)
        confidence = float(np.clip(np.random.normal(CONF_MU[emotion], 0.10), 0.20, 0.99))

        rows.append({
            'interaction_id':          i + 1,
            'student_profile':         pk,
            'situation':               situation,
            'emotion_category':        emotion,
            'message':                 message,
            'zdp_level':               zdp_level,
            'mediation_strategy':      strategy,
            'chatbot_response':        RESPONSES[emotion][zdp_level],
            'zdp_classification':      _zdp_classification(profile),
            'adaptive_progress':       progress,
            'satisfaction':            round(satisfaction, 2),
            'confidence':              round(confidence, 4),
            'interaction_duration_s':  random.randint(30, 300),
        })

    return pd.DataFrame(rows)


# ── Gera os dois pools ────────────────────────────────────────────────────────
print('Gerando Pool A (treino) ...')
train_df = generate_interactions(SIM['n_train'], TEMPLATES_TR)

print('Gerando Pool B (teste) ...')
test_df  = generate_interactions(SIM['n_test'],  TEMPLATES_TE)

print(f'Pool A: {len(train_df)} interações | Pool B: {len(test_df)} interações')
print('\nAmostra Pool A:')
display(train_df[['emotion_category','zdp_level','mediation_strategy','satisfaction','adaptive_progress']].head(5))


Gerando Pool A (treino) ...
Gerando Pool B (teste) ...
Pool A: 800 interações | Pool B: 200 interações

Amostra Pool A:


,emotion_category,zdp_level,mediation_strategy,satisfaction,adaptive_progress
0,frustration,intermediate,guided_problem_solving,3.84,0.5925
1,calm,advanced,guided_practice_incremental_challenge,4.86,0.7794
2,frustration,initial,emotional_support_validation,4.04,0.3712
3,anxiety,advanced,cognitive_restructuring,4.35,0.7143
4,anxiety,intermediate,self_regulation_gradual_steps,4.57,0.5959


## Célula 4 — Dataset Piloto Real (n=25)
Interações coletadas manualmente em português, representando linguagem natural coloquial.
Os textos e labels estão definidos no `vygotea_config.json` (chave `pilot_data`).


In [12]:
pilot_df = pd.DataFrame(cfg['pilot_data']).rename(
    columns={'text': 'message', 'label': 'emotion_category'}
)

print(f'Pilot dataset: {len(pilot_df)} interações')
print('\nDistribuição de emoções no piloto:')
print(pilot_df['emotion_category'].value_counts().to_string())
print('\nAmostra:')
display(pilot_df[['id','emotion_category','zdp','note','message']].head(6))


Pilot dataset: 25 interações

Distribuição de emoções no piloto:
emotion_category
frustration    7
sadness        5
joy            4
anxiety        4
fear           3
calm           2

Amostra:


,id,emotion_category,zdp,note,message
0,R01,anxiety,initial,"explicit anxiety, exam context","Estou muito ansioso com a prova de amanhã, não..."
1,R02,anxiety,intermediate,"social anxiety, performance context",Fico nervoso quando tem apresentação na frente...
2,R03,anxiety,initial,"routine disruption, implicit anxiety","Mudaram o horário sem avisar, agora não sei ma..."
3,R04,anxiety,intermediate,matches Int 1 from paper Table 6,"O projeto em grupo me deixa muito tenso, não s..."
4,R05,fear,intermediate,fear of public judgment,Tenho medo de errar e todo mundo rir de mim na...
5,R06,frustration,initial,Int 3 — implicit frustration via sensory overload,"Não aguento mais o barulho do recreio, preciso..."


## Célula 5 — Treinamento e Comparação de Classificadores
Sete modelos avaliados em Pool B (sintético) e no piloto real.
O modelo proposto no artigo é **TF-IDF + LR ★**.


In [13]:
# ── Keyword Baseline ─────────────────────────────────────────────────────────
class KeywordBaseline:
    """Classificador baseline por correspondência de palavras-chave (paper §3.7)."""
    def fit(self, X, y): return self
    def predict(self, X: List[str]) -> List[str]:
        preds = []
        for msg in X:
            ml = msg.lower()
            counts = {cat: sum(kw in ml for kw in kws) for cat, kws in KW_DICT.items()}
            best = max(counts, key=counts.get)
            preds.append(best if counts[best] > 0 else 'anxiety')
        return preds


# ── Definição do zoo de modelos ───────────────────────────────────────────────
tfidf_base  = dict(max_features=1000, ngram_range=(1, 2), sublinear_tf=True)
tfidf_large = dict(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)

MODELS = {
    'Keyword Baseline': KeywordBaseline(),
    'TF-IDF + NB':      Pipeline([('v', TfidfVectorizer(**tfidf_base)), ('c', ComplementNB(alpha=0.1))]),
    'TF-IDF + SVM':     Pipeline([('v', TfidfVectorizer(**tfidf_base)), ('c', LinearSVC(C=1.0, max_iter=2000, random_state=42))]),
    'TF-IDF + RF':      Pipeline([('v', TfidfVectorizer(**tfidf_base)), ('c', RandomForestClassifier(n_estimators=200, random_state=42))]),
    'TF-IDF + MLP':     Pipeline([('v', TfidfVectorizer(**tfidf_base)), ('c', MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=500, random_state=42))]),
    'LSA(100)+LR':      Pipeline([('v', TfidfVectorizer(**tfidf_large)), ('s', TruncatedSVD(100, random_state=42)), ('n', Normalizer()), ('c', LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=42))]),
    'TF-IDF + LR ★':   Pipeline([('v', TfidfVectorizer(**tfidf_base)), ('c', LogisticRegression(C=1.0, solver='lbfgs', max_iter=500, random_state=42))]),
}

# ── Treinamento e avaliação ───────────────────────────────────────────────────
X_tr = train_df['message'].tolist();  y_tr = train_df['emotion_category'].tolist()
X_te = test_df['message'].tolist();   y_te = test_df['emotion_category'].tolist()
X_pi = pilot_df['message'].tolist();  y_pi = pilot_df['emotion_category'].tolist()

rows_synth = []; rows_pilot = []; cv_skip = {'Keyword Baseline', 'TF-IDF + RF'}

for name, model in MODELS.items():
    print(f'  Treinando {name} ...')
    model.fit(X_tr, y_tr)

    yp_te = model.predict(X_te)
    yp_pi = model.predict(X_pi)

    cv_mean, cv_std = (None, None)
    if name not in cv_skip:
        cv_s = cross_val_score(model, X_tr, y_tr,
                               cv=StratifiedKFold(5, shuffle=True, random_state=42),
                               scoring='f1_macro')
        cv_mean, cv_std = round(cv_s.mean(), 3), round(cv_s.std(), 3)

    rows_synth.append({'Model': name,
                       'Macro F1': round(f1_score(y_te, yp_te, average='macro', zero_division=0), 3),
                       'Accuracy': round(accuracy_score(y_te, yp_te), 3),
                       'CV F1 mean': cv_mean, 'CV F1 std': cv_std})
    rows_pilot.append({'Model': name,
                       'Macro F1': round(f1_score(y_pi, yp_pi, average='macro', zero_division=0), 3),
                       'Accuracy': round(accuracy_score(y_pi, yp_pi), 3)})

comparison_synth = pd.DataFrame(rows_synth)
comparison_pilot = pd.DataFrame(rows_pilot)

print('\n--- Avaliação no Pool B (sintético) ---')
display(comparison_synth)
print('\n--- Avaliação no Piloto Real ---')
display(comparison_pilot)


  Treinando Keyword Baseline ...
  Treinando TF-IDF + NB ...
  Treinando TF-IDF + SVM ...
  Treinando TF-IDF + RF ...
  Treinando TF-IDF + MLP ...
  Treinando LSA(100)+LR ...
  Treinando TF-IDF + LR ★ ...

--- Avaliação no Pool B (sintético) ---


,Model,Macro F1,Accuracy,CV F1 mean,CV F1 std
0,Keyword Baseline,0.296,0.435,NaN,NaN
1,TF-IDF + NB,0.430,0.470,1.0,0.0
2,TF-IDF + SVM,0.394,0.455,1.0,0.0
3,TF-IDF + RF,0.242,0.275,NaN,NaN
4,TF-IDF + MLP,0.426,0.480,1.0,0.0
5,LSA(100)+LR,0.458,0.540,1.0,0.0
6,TF-IDF + LR ★,0.452,0.550,1.0,0.0



--- Avaliação no Piloto Real ---


,Model,Macro F1,Accuracy
0,Keyword Baseline,0.503,0.44
1,TF-IDF + NB,0.080,0.16
2,TF-IDF + SVM,0.121,0.24
3,TF-IDF + RF,0.094,0.16
4,TF-IDF + MLP,0.117,0.24
5,LSA(100)+LR,0.121,0.24
6,TF-IDF + LR ★,0.121,0.24


## Célula 6 — Análise Por Classe (TF-IDF + LR Proposto)


In [14]:
proposed_model = MODELS['TF-IDF + LR ★']
yp_te_lr = proposed_model.predict(X_te)

report = classification_report(y_te, yp_te_lr, output_dict=True, zero_division=0)
per_class_rows = []
for em in EMOTION_DIST:
    if em in report:
        per_class_rows.append({
            'Emotion':    em.capitalize(),
            'Precision':  round(report[em]['precision'], 3),
            'Recall':     round(report[em]['recall'], 3),
            'F1':         round(report[em]['f1-score'], 3),
            'Support':    int(report[em]['support']),
        })

per_class_df = pd.DataFrame(per_class_rows)
print('Per-class breakdown — TF-IDF + LR (Pool B):')
display(per_class_df)
print(f"\nMacro F1: {per_class_df['F1'].mean():.3f}")


Per-class breakdown — TF-IDF + LR (Pool B):


,Emotion,Precision,Recall,F1,Support
0,Anxiety,0.727,0.604,0.660,53
1,Frustration,0.737,0.622,0.675,45
2,Calm,0.622,0.667,0.644,42
3,Sadness,0.333,0.600,0.429,30
4,Joy,1.000,0.056,0.105,18
5,Fear,0.167,0.250,0.200,12



Macro F1: 0.452


## Célula 7 — Análise do Mecanismo ZPD
Avaliamos a distribuição das 1.000 interações entre as três regiões de desenvolvimento.


In [15]:
all_df = pd.concat([train_df, test_df], ignore_index=True)

zdp_counts = all_df['zdp_classification'].value_counts()
total = len(all_df)

zdp_summary = pd.DataFrame([
    {'Classificação': k, 'n': int(zdp_counts.get(k, 0)),
     '%': round(zdp_counts.get(k, 0) / total * 100, 1)}
    for k in ['within_ZPD', 'NDR', 'out_of_ZPD']
])
within_ndr_pct = zdp_summary[zdp_summary['Classificação'].isin(['within_ZPD','NDR'])]['%'].sum()

print('ZPD Classification Summary:')
display(zdp_summary)
print(f'\nCombinado within+NDR: {within_ndr_pct:.1f}%')


ZPD Classification Summary:


,Classificação,n,%
0,within_ZPD,584,58.4
1,NDR,137,13.7
2,out_of_ZPD,279,27.9



Combinado within+NDR: 72.1%


## Célula 8 — Efetividade das Estratégias de Mediação


In [16]:
strategy_eff = (
    all_df.groupby('mediation_strategy')['satisfaction']
    .agg(Mean_Satisfaction='mean', SD='std', N='count')
    .reset_index()
    .sort_values('Mean_Satisfaction', ascending=False)
    .round(3)
)

print('Top-8 estratégias por satisfação média:')
display(strategy_eff.head(8))
print(f"\nSatisfação global: {all_df['satisfaction'].mean():.3f} ± {all_df['satisfaction'].std():.3f}")


Top-8 estratégias por satisfação média:


,mediation_strategy,Mean_Satisfaction,SD,N
4,emotional_support_desensitization,4.352,0.552,6
7,engagement_meaningful_activity,4.284,0.347,62
2,cognitive_restructuring,4.278,0.345,193
3,emotional_support_active_listening,4.260,0.415,21
6,emotional_support_validation,4.257,0.397,43
11,positive_reinforcement_expansion,4.242,0.389,91
1,behavioral_rehearsal,4.239,0.327,28
12,self_regulation_gradual_steps,4.225,0.364,105



Satisfação global: 4.238 ± 0.368


## Célula 9 — Correlação Progresso × Satisfação
> **Nota interpretativa:** r próximo de zero e p > 0.05 indica que as duas métricas são geradas
> por mecanismos independentes dentro da simulação. Isso é esperado e discutido no artigo §6.5.


In [17]:
r_val, p_val = pearsonr(all_df['adaptive_progress'], all_df['satisfaction'])

print(f'Pearson r = {r_val:.4f}')
print(f'p-value   = {p_val:.4f}')
print(f'Significativo (α=0.05): {p_val < 0.05}')
print('\nInterpretação: correlação estatisticamente não significativa — ver artigo §6.5')


Pearson r = 0.0609
p-value   = 0.0543
Significativo (α=0.05): False

Interpretação: correlação estatisticamente não significativa — ver artigo §6.5


## Célula 10 — Rubrica de Avaliação Qualitativa (Especialistas)
Dados definidos em `vygotea_config.json` → chave `specialist_rubric`.


In [18]:
rubric_df = pd.DataFrame([{
    'Interaction': r['id'],
    'Emot. ID':    r['emot_id'],
    'ZPD Adapt.':  r['zpd_adapt'],
    'SE Dev.':     r['se_dev'],
    'Applicab.':   r['applicab'],
    'Note':        r['note'],
} for r in RUBRIC_CFG['interactions']])

dims = ['Emot. ID', 'ZPD Adapt.', 'SE Dev.', 'Applicab.']
rubric_df['Mean'] = rubric_df[dims].mean(axis=1).round(2)

print(f"Rubrica qualitativa (Kendall's W = {RUBRIC_CFG['kendall_w']}, p < {RUBRIC_CFG['kendall_p']})")
display(rubric_df.drop(columns='Note'))
print(f"\nMédias por dimensão:")
for d in dims:
    print(f'  {d}: {rubric_df[d].mean():.2f}')


Rubrica qualitativa (Kendall's W = 0.74, p < 0.05)


,Interaction,Emot. ID,ZPD Adapt.,SE Dev.,Applicab.,Mean
0,Int 1 (Anxiety),4.3,4.0,4.3,4.7,4.32
1,Int 2 (Explicit frustration),3.7,4.3,3.7,4.0,3.92
2,Int 3 (Implicit frustration),3.0,3.7,3.7,3.7,3.53
3,Int 4 (Joy),4.0,3.7,4.0,4.3,4.00
4,Int 5 (Anxiety),4.0,4.0,4.3,4.3,4.15



Médias por dimensão:
  Emot. ID: 3.80
  ZPD Adapt.: 3.94
  SE Dev.: 4.00
  Applicab.: 4.20


## Célula 11 — Geração das Figuras para Publicação
Gera 6 figuras no diretório `./figures/` em formatos `.pdf` (para LaTeX) e `.png` (para preview).
Cada figura corresponde a uma seção do artigo — ver mapeamento abaixo:

| Figura | Seção do artigo | Conteúdo |
|--------|-----------------|----------|
| Fig 1 | §4.2 — Resultados | Comparação de classificadores (sintético vs real) |
| Fig 2 | §4.3 — Resultados | F1 por classe + Precisão/Recall |
| Fig 3 | §5.2 — Discussão | Gap sintético→real por modelo |
| Fig 4 | §4.5 e §5.4 | Distribuição ZPD + Efetividade das estratégias |
| Fig 5 | §5.5 — Discussão | Correlação progresso×satisfação + Dist. emocional |
| Fig 6 | §5.6 — Discussão | Heatmap rubrica qualitativa de especialistas |


In [20]:
os.makedirs('./figures', exist_ok=True)

# Paleta editorial consistente
C_BLUE, C_ORANGE, C_GREEN, C_RED = '#1B4F8A', '#E07B39', '#2E8B57', '#B03A2E'
C_PURPLE, C_TEAL, C_GREY = '#6C3483', '#148F77', '#808B96'
EMOTION_COLORS = {'anxiety':C_BLUE,'frustration':C_ORANGE,'sadness':C_PURPLE,
                  'joy':C_GREEN,'fear':C_RED,'calm':C_TEAL}

plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,
                     'axes.spines.top':False,'axes.spines.right':False,
                     'axes.linewidth':0.8,'figure.dpi':180})

model_names = list(comparison_synth['Model'])
short_names = [n.replace('TF-IDF + ','').replace(' ★','*').replace('Keyword ','KW-') for n in model_names]
f1_synth_vals = comparison_synth['Macro F1'].tolist()
f1_pilot_vals = comparison_pilot['Macro F1'].tolist()

def savefig(name):
    plt.tight_layout()
    plt.savefig(f'./figures/{name}.pdf', bbox_inches='tight')
    plt.savefig(f'./figures/{name}.png', bbox_inches='tight', dpi=200)
    plt.close()
    print(f'  Salvo: ./figures/{name}.pdf + .png')


# ── Figura 1: Comparação de classificadores ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('Figure 1 — Emotion Classifier Evaluation: Synthetic vs Real-World Performance',
             fontsize=11, fontweight='bold', y=1.02)
x = np.arange(len(model_names))
for ax, vals, title, best_col in [
    (axes[0], f1_synth_vals, '(a) Held-out Synthetic Pool (n=200)', C_BLUE),
    (axes[1], f1_pilot_vals, '(b) Real-world Pilot Dataset (n=25)',  C_ORANGE),
]:
    colors = [best_col if v == max(vals) else C_GREY for v in vals]
    bars = ax.bar(x, vals, width=0.6, color=colors, edgecolor='white', zorder=3)
    ax.set_xticks(x); ax.set_xticklabels(short_names, rotation=35, ha='right', fontsize=8)
    ax.set_ylim(0, max(vals)*1.4 + 0.05); ax.set_ylabel('Macro F1', fontsize=9)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.grid(axis='y', linestyle='--', linewidth=0.4, alpha=0.6, zorder=0)
    ax.axhline(max(vals), color=C_RED, ls=':', lw=1.2, label=f'Best={max(vals):.2f}')
    ax.legend(fontsize=7.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.005, f'{v:.2f}', ha='center', fontsize=7.5, fontweight='bold')
savefig('fig1_classifier_comparison')


# ── Figura 2: Per-class F1 ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Figure 2 — Per-class Emotion Analysis (TF-IDF + LR, Held-out Synthetic Pool)',
             fontsize=11, fontweight='bold', y=1.02)
emotions  = per_class_df['Emotion'].str.lower().tolist()
f1s  = per_class_df['F1'].tolist()
prec = per_class_df['Precision'].tolist()
rec  = per_class_df['Recall'].tolist()
cols = [EMOTION_COLORS[e] for e in emotions]
y_pos = np.arange(len(emotions))
axes[0].barh(y_pos, f1s, color=cols, edgecolor='white')
axes[0].set_yticks(y_pos); axes[0].set_yticklabels([e.capitalize() for e in emotions], fontsize=9)
axes[0].axvline(np.mean(f1s), color='black', ls='--', lw=1, label=f'Macro mean={np.mean(f1s):.2f}')
axes[0].set_title('(a) Per-class F1', fontsize=9, fontweight='bold'); axes[0].legend(fontsize=7.5)
for b, v in zip(axes[0].patches, f1s): axes[0].text(v+0.01, b.get_y()+b.get_height()/2, f'{v:.2f}', va='center', fontsize=8)
w = 0.28; xi = np.arange(len(emotions))
axes[1].bar(xi-w, prec, w, label='Precision', color=C_BLUE, alpha=0.85, edgecolor='white')
axes[1].bar(xi,   rec,  w, label='Recall',    color=C_ORANGE, alpha=0.85, edgecolor='white')
axes[1].bar(xi+w, f1s,  w, label='F1',        color=C_GREEN, alpha=0.85, edgecolor='white')
axes[1].set_xticks(xi); axes[1].set_xticklabels([e.capitalize() for e in emotions], rotation=30, ha='right', fontsize=8)
axes[1].set_title('(b) Precision / Recall / F1', fontsize=9, fontweight='bold'); axes[1].legend(fontsize=8)
savefig('fig2_per_class_analysis')


# ── Figura 3: Gap sintético → real ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4.5))
fig.suptitle('Figure 3 — Synthetic-to-Real Generalisation Gap per Classifier', fontsize=11, fontweight='bold')
w = 0.35
ax.bar(x-w/2, f1_synth_vals, w, label='Synthetic Pool B', color=C_BLUE, alpha=0.85, edgecolor='white')
ax.bar(x+w/2, f1_pilot_vals, w, label='Real Pilot',       color=C_ORANGE, alpha=0.85, edgecolor='white')
for xi_, vs, vp in zip(x, f1_synth_vals, f1_pilot_vals):
    delta = vp - vs; col = C_GREEN if delta >= 0 else C_RED
    ax.text(xi_, max(vs,vp)+0.03, f'Δ={delta:+.2f}', ha='center', fontsize=7, color=col, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(short_names, rotation=35, ha='right', fontsize=8.5)
ax.set_ylim(0, 0.75); ax.set_ylabel('Macro F1', fontsize=9)
ax.legend(fontsize=8.5); ax.grid(axis='y', linestyle='--', linewidth=0.4, alpha=0.5)
savefig('fig3_synth_real_gap')


# ── Figura 4: ZPD donut + estratégias ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Figure 4 — ZPD Adaptive Mechanism', fontsize=11, fontweight='bold', y=1.02)
sizes  = [zdp_summary[zdp_summary['Classificação']==k]['%'].values[0] for k in ['within_ZPD','NDR','out_of_ZPD']]
wedges, _, autotexts = axes[0].pie(sizes, labels=['Within ZPD','NDR','Out of ZPD'],
    autopct='%1.1f%%', colors=[C_GREEN, C_BLUE, C_RED], startangle=90,
    textprops={'fontsize':9}, wedgeprops=dict(width=0.5, edgecolor='white', linewidth=1.5))
for at in autotexts: at.set_fontweight('bold')
axes[0].text(0, 0, f'{within_ndr_pct:.1f}%\nwithin+NDR', ha='center', va='center',
             fontsize=10, fontweight='bold')
axes[0].set_title('(a) ZPD Classification (n=1,000)', fontsize=9, fontweight='bold')
eff8 = strategy_eff.head(8).sort_values('Mean_Satisfaction')
short_s = [s.replace('_','\n') for s in eff8['mediation_strategy']]
axes[1].barh(np.arange(len(eff8)), eff8['Mean_Satisfaction'], xerr=eff8['SD'].fillna(0),
             capsize=3, color=C_PURPLE, alpha=0.8, edgecolor='white')
axes[1].set_yticks(np.arange(len(eff8))); axes[1].set_yticklabels(short_s, fontsize=7)
axes[1].axvline(all_df['satisfaction'].mean(), color=C_GREY, ls='--', lw=1.2,
                label=f"Global mean={all_df['satisfaction'].mean():.2f}")
axes[1].set_xlim(3.5, 5.0); axes[1].set_xlabel('Mean Satisfaction', fontsize=9)
axes[1].set_title('(b) Top-8 Strategies by Satisfaction', fontsize=9, fontweight='bold')
axes[1].legend(fontsize=8)
savefig('fig4_zpd_strategy')


# ── Figura 5: Correlação + distribuição emocional ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('Figure 5 — Engagement Dynamics and Emotional Distribution', fontsize=11, fontweight='bold', y=1.02)
for em, grp in all_df.groupby('emotion_category'):
    axes[0].scatter(grp['adaptive_progress'], grp['satisfaction'],
                    alpha=0.18, s=10, color=EMOTION_COLORS[em], label=em.capitalize())
m, b = np.polyfit(all_df['adaptive_progress'], all_df['satisfaction'], 1)
xl = np.linspace(all_df['adaptive_progress'].min(), all_df['adaptive_progress'].max(), 100)
axes[0].plot(xl, m*xl+b, 'k--', lw=1.5, label=f'r={r_val:.3f}, p={p_val:.3f}')
axes[0].set_xlabel('Adaptive Progress $P_t$', fontsize=9); axes[0].set_ylabel('Satisfaction', fontsize=9)
axes[0].set_title(f'(a) Progress vs Satisfaction (not significant)', fontsize=9, fontweight='bold')
axes[0].legend(fontsize=7, ncol=2)
dist = all_df['emotion_category'].value_counts()
target_n = {k: v*len(all_df) for k,v in EMOTION_DIST.items()}
xi = np.arange(len(dist)); w=0.35
axes[1].bar(xi-w/2, dist.values, w, color=[EMOTION_COLORS[e] for e in dist.index], alpha=0.85, label='Observed', edgecolor='white')
axes[1].bar(xi+w/2, [target_n[e] for e in dist.index], w, color=[EMOTION_COLORS[e] for e in dist.index], alpha=0.35, label='Target', hatch='//', edgecolor='white')
axes[1].set_xticks(xi); axes[1].set_xticklabels([e.capitalize() for e in dist.index], rotation=30, ha='right', fontsize=8.5)
axes[1].set_ylabel('Count', fontsize=9); axes[1].set_title('(b) Emotional Distribution', fontsize=9, fontweight='bold')
axes[1].legend(fontsize=8)
savefig('fig5_progress_emotion_dist')


# ── Figura 6: Heatmap rubrica especialistas ───────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
fig.suptitle("Figure 6 — Specialist Qualitative Rubric (Kendall's W=0.74, p<0.05)",
             fontsize=11, fontweight='bold')
dims = ['Emot. ID', 'ZPD Adapt.', 'SE Dev.', 'Applicab.']
heat = rubric_df[dims].values.astype(float)
im = ax.imshow(heat, cmap='YlGn', vmin=2.5, vmax=5.0, aspect='auto')
ax.set_xticks(range(4)); ax.set_xticklabels(dims, fontsize=10)
ax.set_yticks(range(len(rubric_df))); ax.set_yticklabels(rubric_df['Interaction'], fontsize=9)
for i in range(len(rubric_df)):
    for j in range(4):
        v = heat[i, j]; col = 'white' if v > 4.2 else 'black'
        ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=11, fontweight='bold', color=col)
plt.colorbar(im, ax=ax, shrink=0.85, label='Score (1–5)')
savefig('fig6_specialist_rubric')

print('\nTodas as figuras geradas em ./figures/')


  Salvo: ./figures/fig1_classifier_comparison.pdf + .png
  Salvo: ./figures/fig2_per_class_analysis.pdf + .png
  Salvo: ./figures/fig3_synth_real_gap.pdf + .png
  Salvo: ./figures/fig4_zpd_strategy.pdf + .png
  Salvo: ./figures/fig5_progress_emotion_dist.pdf + .png
  Salvo: ./figures/fig6_specialist_rubric.pdf + .png

Todas as figuras geradas em ./figures/


## Célula 12 — Demo: Interação Individual
Testa o pipeline completo com uma mensagem em português brasileiro.


In [21]:
demo_message = (
    "Não consigo dormir pensando na apresentação de amanhã, "
    "fico nervoso só de imaginar todo mundo olhando pra mim."
)
demo_profile = PROFILES['P1']

# Classifica usando o modelo já treinado
demo_emotion  = proposed_model.predict([demo_message])[0]
demo_proba    = proposed_model.predict_proba([demo_message])[0] if hasattr(proposed_model, 'predict_proba') else None
demo_conf     = round(float(demo_proba.max()), 3) if demo_proba is not None else 'N/A'
demo_zdp      = demo_profile['n']
demo_strategy = MEDIATION[demo_emotion][demo_zdp]
demo_response = RESPONSES[demo_emotion][demo_zdp]

print('=' * 60)
print('VygotEA — Demo de Interação Individual')
print('=' * 60)
print(f'Mensagem   : {demo_message}')
print(f'Emoção     : {demo_emotion}  (confiança={demo_conf})')
print(f'Perfil     : {demo_profile["description"]}')
print(f'Nível ZPD  : {demo_zdp}')
print(f'Estratégia : {demo_strategy}')
print(f'Resposta   : {demo_response}')
print('=' * 60)


VygotEA — Demo de Interação Individual
Mensagem   : Não consigo dormir pensando na apresentação de amanhã, fico nervoso só de imaginar todo mundo olhando pra mim.
Emoção     : anxiety  (confiança=0.226)
Perfil     : Student with high social anxiety and restricted interest in technology
Nível ZPD  : intermediate
Estratégia : self_regulation_gradual_steps
Resposta   : I understand your anxiety. How about we use a self-regulation technique?
